In [ ]:
import os
import shutil
import importlib
import copy
import glob
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
import torch
import torchaudio

import phaselocknet_model
import util
import util_localization_psychophysics

import pyfar as pf
from scipy.io import loadmat
from scipy.signal import resample_poly

# Reload modules to reflect recent edits
importlib.reload(phaselocknet_model)
importlib.reload(util)
importlib.reload(util_localization_psychophysics)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

%matplotlib widget

## Model loading

Model is loaded based on the checkpoints, downloaded from https://drive.google.com/drive/folders/1XiS2Q54yH-qblfq4LdmcFy7hl6TmTKHN?usp=share_link 

In [ ]:
"""
Construct example torch model and load weights from checkpoint

* * * NOTE: This cell requires torch checkpoint to have been downloaded.
To convert tensorflow checkpoints to torch checkpoints, see cells below.
"""
# dir_model = "models/sound_localization/simplified_IHC3000/arch01"
dir_model = "models/sound_localization/simplified_IHC3000_delayed_integration/arch01"

# # Option 1: Manually construct the model from config files:
# with open(os.path.join(dir_model, "config.json"), "r") as f:
#     config_model = json.load(f)
# with open(os.path.join(dir_model, "arch.json"), "r") as f:
#     architecture = json.load(f)
# model = phaselocknet_model.Model(
#     config_model=config_model,
#     architecture=architecture,
#     input_shape=[2, 60000, 2],  # <-- [batch, timesteps @ 50 kHz sampling rate, channels==2] for sound_localization
#     config_random_slice={"size": [50, 10000], "buffer": [0, 500]},
# )

# Option 2: Use the `phaselocknet_model.get_model` function
model, config_model = phaselocknet_model.get_model(
    dir_model=dir_model,
    fn_config="config.json",
    fn_arch="arch.json",
)

# Load model weights from torch checkpoint
util.load_model_checkpoint(
    model=model.perceptual_model,
    dir_model=dir_model,
    fn_ckpt="ckpt_BEST.pt",
    weights_only=True,
)

model.train(mode=False)
model.to(device)
assert not model.training

model


## Stimulus generation
A wavelet of white noise is generated and binauralized with a HRTF from the SOFA file of SADIE database. One desired azimuth angle is chosen. The stimulus is sampled at 50 kHz and has a duration of 1.2 s, following the model setup. The wavelet is made using a centered Hanning window of 0.6 s, so that the stimulus has a smooth onset and offset.

In [ ]:
# Desired azimuth angle for stimulus generation in degrees (e.g., 10 degrees to the left of the listener)
desired_azimuth_deg = -90

# Load HRTFs with pyfar and binauralize a 1.2 s white-noise burst at 50 kHz
sofa_path = "HRTFs/D1_48K_24bit_256tap_FIR_SOFA.sofa"
hrirs, source_coords, _ = pf.io.read_sofa(sofa_path)

# Select nearest source direction to azimuth=+90 deg (left), elevation=0 deg
# Option 1: Use pyfar's built-in nearest neighbor search with spherical distance
desired_coords = pf.Coordinates.from_spherical_elevation(azimuth=np.deg2rad(desired_azimuth_deg), elevation=0, radius=1.2)
desired_idx, _ = source_coords.find_nearest(desired_coords, distance_measure="spherical_radians")
idx = desired_idx[0]
# Option 2: Manually compute nearest neighbor in azimuth/elevation space
# angles = source_coords.spherical_elevation  # [azimuth, elevation, radius] in radians
# angles_deg = np.column_stack([np.rad2deg(angles[:, 0]), np.rad2deg(angles[:, 1])])
# target_deg = np.array([90.0, 0.0])
# idx = int(np.argmin(np.sum((angles_deg - target_deg) ** 2, axis=1)))
# print(f"Nearest HRTF direction (deg): az={angles_deg[idx, 0]:.1f}, el={angles_deg[idx, 1]:.1f}")
print(f"Nearest available HRTF direction (deg): az={source_coords.azimuth[idx]:.1f}, el={source_coords.elevation[idx]:.1f}")

# HRIR shape is [ears, taps]
hrir = hrirs.time[idx]
if hrir.shape[0] != 2 and hrir.shape[-1] == 2:
    hrir = hrir.T

# Generate white-noise burst at 50 kHz, 1.2 s -> 60000 samples
sr = 50000
n_samples = 60000
rng = np.random.default_rng(0)
noise = rng.standard_normal(n_samples).astype(np.float32)

# Apply a 0.6 s Hann window centered in the 1.2 s array
win_len = int(0.6 * sr)  # 30000 samples
hann_win = np.hanning(win_len).astype(np.float32)

envelope = np.zeros(n_samples, dtype=np.float32)
start = (n_samples - win_len) // 2
end = start + win_len
envelope[start:end] = hann_win

noise *= envelope

# SOFA HRIRs are 48 kHz, so resample FIRs to 50 kHz for convolution at 50 kHz
hrir_50k = resample_poly(hrir, up=25, down=24, axis=-1)

# Binauralize: convolve mono noise with left/right HRIRs
left = np.convolve(noise, hrir_50k[0], mode="same")
right = np.convolve(noise, hrir_50k[1], mode="same")
binaural = np.stack([left, right], axis=-1).astype(np.float32)  # (60000, 2)

plt.close('all')
# Use an explicit 2D axes so this cell is independent of previously active 3D axes.
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(binaural[:, 0], label="Left ear")
ax.plot(binaural[:, 1], label="Right ear")
ax.set_xlabel("Sample")
ax.set_ylabel("Amplitude")
ax.set_title("Binaural Signal")
ax.legend()
plt.show()

# Final tensor shape: (1, 60000, 2), last dim is ears
x = torch.from_numpy(binaural).unsqueeze(0)
print("Binaural signal input shape (batch x time x channels):", tuple(x.shape), ", dtype:", x.dtype)

## Model Evaluation
The binaural signal is fed into the model. The output is a tensor of shape (batch, 504), where 504 are classification probabilities corresponding to each source direction.

In [ ]:
with torch.no_grad():
    out = model(x.to(device))
print("Model output:")
for k, v in out.items():
    print("|__", k, v.shape if v.ndim > 0 else v, v.dtype, np.argmax(v[0].cpu().numpy()))

map_azimuth, map_elevation = util_localization_psychophysics.label_to_azim_elev(torch.argmax(out['label_loc_int']))
print(f"Predicted azimuth: {map_azimuth} degrees")
print(f"Predicted elevation: {map_elevation} degrees")

In [ ]:
plt.close('all')

# Convert logits (1 x 504) to probabilities
logits = out["label_loc_int"][0].cpu()
probs = torch.softmax(logits, dim=0).numpy()

# Map class index -> (azimuth, elevation) in degrees
def _to_scalar(val):
    if torch.is_tensor(val):
        return float(val.cpu().item())
    return float(np.asarray(val).squeeze())

az_deg = np.zeros_like(probs, dtype=np.float64)
el_deg = np.zeros_like(probs, dtype=np.float64)

for i in range(probs.shape[0]):
    az_el = util_localization_psychophysics.label_to_azim_elev(i)
    az_deg[i] = _to_scalar(az_el[0])
    el_deg[i] = _to_scalar(az_el[1])

# Wrap azimuth to [-180, 180)
az_deg = ((az_deg + 180.0) % 360.0) - 180.0

# Spherical (az, el) -> Cartesian on unit sphere
az = np.deg2rad(az_deg)
el = np.deg2rad(el_deg)
xs = np.cos(el) * np.cos(az)
ys = np.cos(el) * np.sin(az)
zs = np.sin(el)

# 3D sphere heatmap (scatter on sphere)
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection="3d")

# Optional wireframe sphere for context
u, v = np.mgrid[0:2*np.pi:50j, -np.pi/2:np.pi/2:25j]
ax.plot_wireframe(np.cos(v)*np.cos(u), np.cos(v)*np.sin(u), np.sin(v),
                  color="lightgray", linewidth=0.4, alpha=0.4)

sc = ax.scatter(xs, ys, zs, c=probs, cmap="YlOrRd", s=25, depthshade=False)
cbar = plt.colorbar(sc, ax=ax, pad=0.08, shrink=0.8)
cbar.set_label("Probability")

imax = int(np.argmax(probs))
ax.scatter(xs[imax], ys[imax], zs[imax], c="red", s=120, marker="*", label=f"max idx={imax}")

ax.set_title("Localization probability heatmap on sphere")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_box_aspect([1, 1, 1])
ax.legend(loc="upper left")
plt.show()

print(f"Max-probability index: {imax}, prob={probs[imax]:.4f}, "
      f"az={az_deg[imax]:.1f}°, el={el_deg[imax]:.1f}°")

In [ ]:
# Save binaural signal for phaselocknet_evaluate_CIAT.py
# Expected CLI usage:
# python phaselocknet_evaluate_CIAT.py --signal-npy stimuli/demo_binaural_signal.npy --signal-sr 50000

signal_out = binaural.astype(np.float32)
np.save("stimuli/demo_binaural_signal.npy", signal_out)

print("Saved demo_binaural_signal.npy", signal_out.shape, signal_out.dtype)

In [ ]:
# Run phaselocknet_evaluate_CIAT.py from inside this notebook
import ast
import os
import re
import subprocess
import sys

signal_path = "stimuli/demo_binaural_signal.npy"
signal_sr = 50000
script_path = "phaselocknet_evaluate_CIAT.py"

if not os.path.exists(signal_path):
    raise FileNotFoundError(
        f"{signal_path} not found. Run the save cell first to create it."
    )

cmd = [
    sys.executable,
    script_path,
    "--signal-npy",
    signal_path,
    "--signal-sr",
    str(signal_sr),
]

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
stdout = result.stdout.strip()
print(stdout)

# Parse scalar OR array output from phaselocknet_evaluate_CIAT.py
m = re.search(
    r"estimated_class_index=(.*?),\s*estimated_angle=(.*?),\s*estimated_angle_expected=(.*)$",
    stdout,
    flags=re.MULTILINE,
)
if m is not None:
    estimated_class_index = ast.literal_eval(m.group(1).strip())
    estimated_angle = ast.literal_eval(m.group(2).strip())
    estimated_angle_expected = ast.literal_eval(m.group(3).strip())

    print(f"Parsed estimated_class_index: {estimated_class_index}")
    print(f"Parsed estimated_angle (argmax azimuth deg): {estimated_angle}")
    print(
        "Parsed estimated_angle_expected (posterior mean azimuth deg): "
        f"{estimated_angle_expected}"
    )
else:
    print("Could not parse estimator output. Raw stdout shown above.")

In [ ]:
# Load MATLAB stimulus array from .mat file
mat_path = "stimuli/llado2026_bin_stim_array.mat"
mat_data = loadmat(mat_path)

# Exclude MATLAB metadata keys and inspect payload variables
payload_keys = [k for k in mat_data.keys() if not k.startswith("__")]
print("Loaded:", mat_path)
print("Variables:", payload_keys)

for k in payload_keys:
    v = mat_data[k]
    shape = getattr(v, "shape", None)
    dtype = getattr(v, "dtype", type(v))
    print(f"  {k}: shape={shape}, dtype={dtype}")

if "mat_data" not in globals():
    raise RuntimeError("Run the MAT loading cell first so `mat_data` is available.")
if "bin_stim_array" not in mat_data:
    raise KeyError("`bin_stim_array` not found in MAT file payload.")
if "fs" not in mat_data:
    raise KeyError("`fs` not found in MAT file payload.")

bin_stim_array = np.asarray(mat_data["bin_stim_array"], dtype=np.float32)
fs = int(np.asarray(mat_data["fs"]).reshape(-1)[0])

print("bin_stim_array shape:", bin_stim_array.shape)
print("fs:", fs)

signal_path = "stimuli/llado2026_bin_stim_array.npy"
np.save(signal_path, bin_stim_array)

In [ ]:
%run phaselocknet_evaluate_CIAT.py --signal-npy stimuli/llado2026_bin_stim_array.npy --signal-sr 48000 --eval-batch-size 3